# Lab 9 — Time series plotting (with station filter)

Run cells in order. Use the characteristic dropdown and the station selector (legend filter) to control which stations are plotted. Each 
 click will add a new interactive Plotly figure below and will not remove previous plots.

In [11]:
# 0) Ensure required packages are available
import importlib, subprocess, sys

def ensure(pkg, import_name=None):
    name = import_name or pkg
    try:
        return importlib.import_module(name)
    except Exception:
        print(f'Installing {pkg}...')
        try:
            subprocess.check_call(
                [sys.executable, '-m', 'pip', 'install', pkg],
                stdout=subprocess.DEVNULL,
                stderr=subprocess.DEVNULL,
            )
        except subprocess.CalledProcessError as e:
            print(f'Unable to install {pkg}: {e}')
            return None
        try:
            return importlib.import_module(name)
        except Exception as e:
            print(f'Import failed for {name} after installation: {e}')
            return None

plotly = ensure('plotly')
ipywidgets = ensure('ipywidgets')
pandas = ensure('pandas')

if plotly is None or pandas is None:
    raise RuntimeError('plotly and pandas are required. install manually and restart.')

if ipywidgets is None:
    print('ipywidgets is not available; interactive widget control will be skipped.')

# optional: enable notebook mode for plotly offline if available
try:
    from plotly.offline import init_notebook_mode
    init_notebook_mode(connected=True)
except Exception:
    pass

In [12]:
# 1) Load narrowresult.csv and build datetime column
import pandas as pd
from IPython.display import display
CSV = 'narrowresult.csv'
df = pd.read_csv(CSV, low_memory=False)
date_col = 'ActivityStartDate'
time_col = 'ActivityStartTime/Time'
if date_col in df.columns:
    if time_col in df.columns:
        df['datetime'] = pd.to_datetime(df[date_col].astype(str).fillna('') + ' ' + df[time_col].astype(str).fillna(''), errors='coerce')
    else:
        df['datetime'] = pd.to_datetime(df[date_col], errors='coerce')
else:
    raise RuntimeError('Required date column not found in CSV')
print('Loaded', CSV, 'rows =', len(df))
display(df.head(2))

Loaded narrowresult.csv rows = 3571


,OrganizationIdentifier,OrganizationFormalName,ActivityIdentifier,ActivityStartDate,ActivityStartTime/Time,ActivityStartTime/TimeZoneCode,MonitoringLocationIdentifier,ResultIdentifier,DataLoggerLine,ResultDetectionConditionText,...,ResultLaboratoryCommentCode,ResultLaboratoryCommentText,ResultDetectionQuantitationLimitUrl,LaboratoryAccreditationIndicator,LaboratoryAccreditationAuthorityName,TaxonomistAccreditationIndicator,TaxonomistAccreditationAuthorityName,LabSamplePreparationUrl,ProviderName,datetime
0,31ORWUNT_WQX,Ohio River Valley Water Sanitation Commission ...,31ORWUNT_WQX-OR640M:20240521:1030:SFST,2024-05-21,10:30:00,EDT,31ORWUNT_WQX-OR640M,STORET-1046729626,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,STORET,2024-05-21 10:30:00
1,31ORWUNT_WQX,Ohio River Valley Water Sanitation Commission ...,31ORWUNT_WQX-OR4495M:20240523:1330:SFST,2024-05-23,13:30:00,EDT,31ORWUNT_WQX-OR4495M,STORET-1046729053,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,STORET,2024-05-23 13:30:00


In [13]:
# 2) Prepare numeric values and list available characteristics and stations
df['ResultMeasureValue_numeric'] = pd.to_numeric(df.get('ResultMeasureValue'), errors='coerce')
char_col = 'CharacteristicName'
station_col = 'MonitoringLocationIdentifier'
if char_col not in df.columns or station_col not in df.columns:
    raise RuntimeError('Required columns missing in CSV')
available_chars = sorted(df[char_col].dropna().unique())
available_stations = sorted(df[station_col].dropna().unique())
print('Found', len(available_chars), 'characteristics and', len(available_stations), 'stations')
print('Example characteristics:', available_chars[:20])
print('Example stations:', available_stations[:10])

Found 74 characteristics and 21 stations
Example characteristics: ['Acidity, (H+)', 'Aluminum', 'Ammonia', 'Ammonium', 'Antimony', 'Arsenic', 'Barium', 'Barometric pressure', 'Beryllium', 'Biochemical oxygen demand, standard conditions', 'Bromide', 'Cadmium', 'Calcium', 'Chloride', 'Chromium', 'Copper', 'Depth', 'Dissolved oxygen (DO)', 'Dissolved oxygen saturation', 'Escherichia coli']
Example stations: ['11NPSWRD_WQX-ABLI_HSSS', '11NPSWRD_WQX-ABLI_KCKC', '11NPSWRD_WQX-ABLI_NBKC', '11NPSWRD_WQX-ABLI_SSTS', '11NPSWRD_WQX-BISO_12', '11NPSWRD_WQX-BISO_13', '11NPSWRD_WQX-BISO_14', '11NPSWRD_WQX-BISO_16', '31ORWUNT_WQX-KY4.1M', '31ORWUNT_WQX-LR-4.5M']


In [14]:
# 3) Plotting function (accepts station filter)
import plotly.graph_objects as go
from IPython.display import display
import math

def plot_characteristic(characteristic, stations=None, df=df, save_html=True):
    """Plot `characteristic` vs time for every station in `stations` (or all if None).
    Appends a Plotly figure to the notebook output.
    """
    sel = df[df[char_col] == characteristic].copy()
    sel['ResultMeasureValue_numeric'] = pd.to_numeric(sel.get('ResultMeasureValue'), errors='coerce')
    sel = sel.dropna(subset=['datetime', 'ResultMeasureValue_numeric'])
    if stations is not None:
        sel = sel[sel[station_col].isin(stations)]
    if sel.empty:
        print('No numeric results found for', characteristic, 'with selected stations')
        return None

    fig = go.Figure()
    grouped = sel.groupby(station_col)
    for name, g in grouped:
        g = g.sort_values('datetime')
        x = g['datetime']
        y = g['ResultMeasureValue_numeric']
        if y.dropna().empty or all([not math.isfinite(v) for v in y.dropna()]):
            continue
        fig.add_trace(go.Scatter(x=x, y=y, mode='lines+markers', name=str(name), hovertemplate='%{text}',
                                 text=[f'Station: {name}<br>{characteristic}: {val}' for val in y]))

    fig.update_layout(title=f'{characteristic} vs time', xaxis_title='Date/Time', yaxis_title=characteristic, height=600, legend=dict(itemsizing='trace'))
    # display the interactive figure inline
    display(fig)
    if save_html:
        safe_name = ''.join(c if c.isalnum() else '_' for c in str(characteristic))
        fname = f'timeseries_{safe_name}.html'
        fig.write_html(fname, include_plotlyjs='cdn')
        print('Saved interactive plot to', fname)
    return fig

In [15]:
# 4) Interactive UI: characteristic dropdown, station multi-select (legend filter), and Plot button
import ipywidgets as widgets
from IPython.display import display, clear_output

char_dropdown = widgets.Dropdown(options=available_chars, description='Characteristic:', layout=widgets.Layout(width='60%'))
station_selector = widgets.SelectMultiple(options=available_stations, description='Stations:', layout=widgets.Layout(width='60%', height='200px'))
select_all_btn = widgets.Button(description='Select all stations')
deselect_all_btn = widgets.Button(description='Deselect all')
save_toggle = widgets.Checkbox(value=True, description='Save HTML')
plot_button = widgets.Button(description='Plot & Append', button_style='primary')

def on_select_all(b):
    station_selector.value = tuple(available_stations)
def on_deselect_all(b):
    station_selector.value = tuple()

def on_plot_clicked(b):
    ch = char_dropdown.value
    selected = list(station_selector.value) if station_selector.value else None
    print('Plotting:', ch, 'Stations count:', len(selected) if selected is not None else 'ALL')
    plot_characteristic(ch, stations=selected, df=df, save_html=bool(save_toggle.value))

select_all_btn.on_click(on_select_all)
deselect_all_btn.on_click(on_deselect_all)
plot_button.on_click(on_plot_clicked)

ui = widgets.VBox([char_dropdown, widgets.HBox([station_selector, widgets.VBox([select_all_btn, deselect_all_btn, save_toggle, plot_button])])])
display(ui)

# Note: clicking a station name in the Plotly legend will also hide/show that trace interactively.

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'hovertemplate': '%{text}',
              'mode': 'lines+markers',
              'name': 'USGS-03254520',
              'text': [Station: USGS-03254520<br>Acidity, (H+): 1e-05, Station:
                       USGS-03254520<br>Acidity, (H+): 1e-05, Station:
                       USGS-03254520<br>Acidity, (H+): 2e-05, Station:
                       USGS-03254520<br>Acidity, (H+): 1e-05, Station:
                       USGS-03254520<br>Acidity, (H+): 1e-05, Station:
                       USGS-03254520<br>Acidity, (H+): 2e-05],
              'type': 'scatter',
              'x': array(['2024-01-17T12:20:00.000000', '2024-01-17T12:30:00.000000',
                          '2024-01-26T12:50:00.000000', '2024-02-06T11:10:00.000000',
                          '2024-02-06T12:40:00.000000', '2024-02-29T10:40:00.000000'],
                         dtype='datetime64[us]'),
              'y': {'bdata': '8WjjiLX45D7xaOOItfjkPvFo44i1+PQ+8WjjiLX45D7xaOOItfjkPvFo44i1+PQ+', 'dtype': 'f8'}},
             {'hovertemplate': '%{text}',
              'mode': 'lines+markers',
              'name': 'USGS-03290500',
              'text': [Station: USGS-03290500<br>Acidity, (H+): 1e-05, Station:
                       USGS-03290500<br>Acidity, (H+): 1e-05],
              'type': 'scatter',
              'x': array(['2024-01-25T11:20:00.000000', '2024-01-25T13:50:00.000000'],
                         dtype='datetime64[us]'),
              'y': {'bdata': '8WjjiLX45D7xaOOItfjkPg==', 'dtype': 'f8'}},
             {'hovertemplate': '%{text}',
              'mode': 'lines+markers',
              'name': 'USGS-03302058',
              'text': [Station: USGS-03302058<br>Acidity, (H+): 1e-05, Station:
                       USGS-03302058<br>Acidity, (H+): 1e-05, Station:
                       USGS-03302058<br>Acidity, (H+): 2e-05, Station:
                       USGS-03302058<br>Acidity, (H+): 2e-05],
              'type': 'scatter',
              'x': array(['2024-01-03T12:30:00.000000', '2024-01-03T12:50:00.000000',
                          '2024-02-29T11:00:00.000000', '2024-02-29T11:40:00.000000'],
                         dtype='datetime64[us]'),
              'y': {'bdata': '8WjjiLX45D7xaOOItfjkPvFo44i1+PQ+8WjjiLX49D4=', 'dtype': 'f8'}}],
    'layout': {'height': 600,
               'legend': {'itemsizing': 'trace'},
               'template': '...',
               'title': {'text': 'Acidity, (H+) vs time'},
               'xaxis': {'title': {'text': 'Date/Time'}},
               'yaxis': {'title': {'text': 'Acidity, (H+)'}}}
})

Notes:
- Use the 'Select all stations' button to include every station in the plot.
- Each 'Plot & Append' click creates a new figure below; earlier figures remain visible.
- Plotly legend supports click-to-hide traces; the station selector filters stations before plotting.